In [ ]:
!pip install groq python-dotenv google-generativeai crewai

In [ ]:
import os
import json
import random
from datetime import datetime, timezone
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

groq_client = Groq(api_key="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")
GROQ_MODEL = "qwen/qwen3-32b"

CONTENT_FORMATS = ["static_post", "carousel", "infographic", "poll", "reel", "tiktok", "youtube_short"]
VIDEO_FORMATS = {"reel", "tiktok", "youtube_short"}
VISUAL_FORMATS = {"carousel", "infographic"}

In [ ]:
SYSTEM_PROMPT = f"""You are a social media content strategist.
Return ONLY a valid JSON object (no markdown, no explanation) with this shape:
{{
  "campaign_name": "string",
  "days": [
    {{
      "day": 1,
      "journey_stage": "string",
      "platform": "string",
      "content_format": "one of {CONTENT_FORMATS}",
      "content_pillar": "string",
      "post_idea": "string",
      "hook": "string, under 12 words",
      "caption": "string, publish-ready",
      "hashtags": ["#tag1", "#tag2"],
      "cta": "string",
      "visual_notes": "string or null (required for carousel/infographic)"
    }}
    ... exactly 7 items, day 1 to 7
  ]
}}

Rules:
- Use a diverse mix of content_format: include at least one carousel/infographic,
  one poll, and one short video format (reel/tiktok/youtube_short).
- Distribute days across the given customer_journey stages.
- Ground every post in the strategy info given below.
"""


def build_user_prompt(strategy: dict, campaign_name: str) -> str:
    personas = strategy["customer_personas"]

    # personas ممكن تكون list فيها كذا برسونة أو dict واحدة - نتعامل مع الحالتين
    if isinstance(personas, dict):
        personas = [personas]

    personas_text = "\n".join(
        f"- {p.get('name', f'Persona {i+1}')}: "
        f"audience={p.get('target_audience')}, "
        f"pain_points={p.get('pain_points')}, "
        f"motivations={p.get('motivations')}"
        for i, p in enumerate(personas)
    )

    # customer_journey ممكن تكون list من stages أو dict {stage: description}
    journey = strategy.get("customer_journey", [])
    if isinstance(journey, dict):
        journey_text = "\n".join(f"- {stage}: {desc}" for stage, desc in journey.items())
        journey_stage_names = list(journey.keys())
    else:
        journey_text = "\n".join(f"- {stage}" for stage in journey)
        journey_stage_names = journey

    return f"""
Campaign name: {campaign_name}

PERSONAS (cover all of them across the week, not just one):
{personas_text}

Marketing angles: {strategy.get('marketing_angles', [])}
Campaign ideas: {strategy.get('campaign_ideas', [])}
SWOT: {strategy.get('swot', {})}

CUSTOMER JOURNEY STAGES TO COVER ACROSS THE 7 DAYS:
{journey_text}

Use these exact stage names in the "journey_stage" field: {journey_stage_names}

Return ONLY the JSON object described in the system prompt.
"""

In [ ]:
def extract_json(raw_text: str) -> dict:
    text = raw_text.strip().strip("`")
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

def clean_calendar(calendar: dict) -> dict:
    for d in calendar.get("days", []):
        d["hashtags"] = [h if h.startswith("#") else f"#{h}" for h in d.get("hashtags", [])]
    return calendar

In [ ]:
def enforce_format_diversity(calendar: dict) -> dict:
    days = calendar["days"]
    formats_used = {d["content_format"] for d in days}
    static_days = [d for d in days if d["content_format"] == "static_post"]

    needed = []
    if not formats_used & VIDEO_FORMATS:
        needed.append(random.choice(list(VIDEO_FORMATS)))
    if not formats_used & VISUAL_FORMATS:
        needed.append(random.choice(list(VISUAL_FORMATS)))
    if "poll" not in formats_used:
        needed.append("poll")

    for fmt, day in zip(needed, static_days):
        day["content_format"] = fmt
        if fmt in VISUAL_FORMATS and not day.get("visual_notes"):
            day["visual_notes"] = f"Break '{day['post_idea']}' into 3-5 slides."

    return calendar

In [ ]:
def generate_content_calendar(strategy: dict, campaign_name: str, max_attempts: int = 2) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(strategy, campaign_name)},
    ]

    last_error = None
    for attempt in range(max_attempts):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                temperature=0.7,
                max_tokens=4096,  # كان الافتراضي صغير وبيقطع الـ JSON الطويل
                response_format={"type": "json_object"},
            )
            calendar = extract_json(response.choices[0].message.content)
            calendar = clean_calendar(calendar)
            calendar = enforce_format_diversity(calendar)

            calendar["campaign_name"] = campaign_name
            calendar["generated_at"] = datetime.now(timezone.utc).isoformat()
            calendar["model_used"] = f"groq/{GROQ_MODEL}"
            return calendar

        except Exception as e:
            last_error = e
            print(f"[WARN] Attempt {attempt + 1} failed: {e}")

    raise last_error

In [ ]:
sample_strategy = {
    "customer_personas": [{
          "name" : "The Budget-Conscious Gen Z",
          "target_audience" : "18-24 years, university students or entry-level jobs.",
          "pain_points": ["High price sensitivity; feels ripped off by generic Amazon sellers with poor quality control."],
          "motivations": ["Looking for the 'sweet spot' between Hala Fashion's low cost and premium brands like Zara."]
      },{
          "name" : "The Value-Seeking Professional",
          "target_audience" : "25-34 years, Cairo/Giza residents, office workers and students.",
          "pain_points": ["Fear of buying cheap shirts that shrink after the first wash or feel itchy against skin."],
          "motivations": ["Needs a reliable daily staple that looks clean for work but costs less than imported brands."]
      }
     ],
    "swot": {
            "strengths": ["Consistent fabric durability compared to mass-market alternatives","Clean, versatile aesthetic suitable for Egyptian weather (AC/heat)", "Mid-range pricing strategy avoids entry-level price wars"],
            "weaknesses": ["Lack of unique branding or graphics makes it easily substitutable","No established loyalty program in a saturated market","Standard cotton weave lacks the 'premium' feel of imported fabrics"],
            "opportunities":  ["Position as 'Egyptian Quality Standard' to counter cheap imports","Bundle with accessories (socks, caps) for higher basket size on Noon/Amazon","Aeverage Ramadan sales events where basics see a spike in demand"],
            "threats": ["Aggressive discounting by Hala Fashion during end-of-season clearances","Economic inflation driving consumers toward the absolute lowest price point (<EGP 200)","Rise of local fast-fashion brands offering similar specs at lower costs"]
        },
    "marketing_angles": [
        {"angle_name": "The Reliable Staple", "hook_message": "Stop shrinking. Start wearing the black tee tha"},
        {"angle_name": "Wardrobe Essentialist", "hook_message": "One shirt, 10 outfits: The only thing you need"},
        {"angle_name": "Quality vs. Price Truth", "hook_message": "Why paying EGP 250 is a waste when this fits t"}
     ],

    "campaign_ideas": [
        {"campaign_name": "#TheBlack Tee Truth", "concept": "Unboxing video series where influencers wash"},
        {"campaign_name": "Office Ready Essentials", "concept": "Flat-lay photos styling the black tee with Egypt"}
    ],

    "customer_journey": {
        "awareness":"User sees a Reel on TikTok showing the shirt's texture close-u p, contrasting it with a cheap alternative that shrinks.",
        "consideration":"They click to see detailed specs and read reviews specifically mentioning 'wash durability' rather than just price.",
        "conversion":"Purchase via Noon or Amazon using a limited-time coupon code (e.g., 'EGYPTIANQUALITY') for EGP 290.",
        "retention":"Encourage repurchase by offering free shipping on the second o rder and asking for feedback on fit consistency."
    },
    "success_kpis": ["Conversion Rate on Noon/Amazon(Target: >2.5%)", "Return Rate due to sizing or qualityissues (<3%)"]
}



calendar = generate_content_calendar(sample_strategy, "15 Minutes to Dinner")
print(json.dumps(calendar, ensure_ascii=False, indent=2))

{
  "campaign_name": "15 Minutes to Dinner",
  "days": [
    {
      "day": 1,
      "journey_stage": "awareness",
      "platform": "tiktok",
      "content_format": "reel",
      "content_pillar": "Quality vs. Price Truth",
      "post_idea": "Side-by-side unboxing of the black tee vs. a cheap alternative, showing shrinkage after washing.",
      "hook": "Why paying EGP 250 is a waste when this fits like new after 5 washes?",
      "caption": "Don’t get scammed by cheap tees that shrink! Our black tee keeps its shape and quality. 💯 #EgyptianQuality #TheBlackTeeTruth",
      "hashtags": [
        "#TheBlackTeeTruth",
        "#EGYPTIANQUALITY",
        "#NoMoreShrinking"
      ],
      "cta": "Watch the full test and shop now!",
      "visual_notes": "Split-screen video: left side shows a generic tee shrinking in a washing machine, right side shows the target shirt remaining unchanged."
    },
    {
      "day": 2,
      "journey_stage": "consideration",
      "platform": "instagram",